In [1]:
# -------------------------------
# Video Frame Point Annotator (Tkinter)
# Python 3.8+ | Jupyter compatible
# Adds: specific frame selection (1-based), new JSON schema (per frame),
#       "Delete Saved (this frame)" button, and robust invalid-frame handling
# Keeps: your UI layout, zoom system, canvas drawing, shortcuts, next/prev
# -------------------------------

import json
import os
from pathlib import Path
import sys
import traceback

import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
from PIL import Image, ImageTk

import re
from tkinter import ttk

# --- Backends for video frame extraction ---
def read_frame(video_path: Path, frame_id_1based: int):
    """
    Return a specific frame of a video as an RGB numpy array (H, W, 3).
    frame_id_1based: 1,2,3,...  (1-based)
    Tries OpenCV first; if not available/failed, tries imageio.
    Raises RuntimeError if fails.
    """
    if frame_id_1based is None or frame_id_1based < 1:
        raise RuntimeError(f"Invalid frame id: {frame_id_1based}")

    # Try OpenCV (cv2)
    try:
        import cv2
        cap = cv2.VideoCapture(str(video_path))
        if not cap or not cap.isOpened():
            raise RuntimeError("OpenCV failed to open video")
        # Seek to 0-based index
        target_idx0 = int(frame_id_1based) - 1
        # Set position and read
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx0)
        ok, frame = cap.read()
        cap.release()
        if ok and frame is not None:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            return frame
        else:
            raise RuntimeError(f"OpenCV failed to read frame {frame_id_1based}")
    except Exception:
        pass  # fallback to imageio

    # Try imageio
    try:
        import imageio.v2 as imageio
        reader = imageio.get_reader(str(video_path))
        frame = reader.get_data(int(frame_id_1based) - 1)  # zero-based index
        reader.close()
        return frame
    except Exception as e:
        raise RuntimeError(f"Failed to read frame {frame_id_1based} via OpenCV and imageio: {e}")


class VideoAnnotator(tk.Tk):
    def __init__(self, video_dir, margin=(60, 160), min_display_size=(320, 240), fit_scale_cap=1.0):
        """
        :param video_dir: path containing videos + object_specify_frames.json
        :param margin: (w_margin, h_margin) deducted from screen size for fitting UI
        :param min_display_size: lower bound for display size after scaling
        :param fit_scale_cap: limit the automatic fit scale to this (<=1.0 means never upscale)
        """
        super().__init__()
        self.title("Video Frame Annotator")

        self.video_dir = Path(video_dir).expanduser().resolve()
        # NEW: per-frame json filename (no conflict with old)
        self.json_path = self.video_dir / "object_specify_frames.json"

        # UI fit parameters
        self.screen_w = self.winfo_screenwidth()
        self.screen_h = self.winfo_screenheight()
        self.w_margin, self.h_margin = margin
        self.avail_w = max(200, self.screen_w - self.w_margin)  # available width for image
        self.avail_h = max(200, self.screen_h - self.h_margin)  # available height for image
        self.min_disp_w, self.min_disp_h = min_display_size
        self.fit_scale_cap = float(fit_scale_cap)

        # Zoom (relative to fit). 1.0 = fit-to-screen. User can set 0.2~1.5 via slider.
        self.zoom_var = tk.DoubleVar(value=1.0)

        # Supported video extensions
        self.extensions = {".avi", ".mp4", ".mov", ".mkv", ".m4v", ".wmv", ".flv", ".webm", ".mpg", ".mpeg"}

        # Load videos
        self.video_list = self._find_videos()
        if not self.video_list:
            messagebox.showerror("No videos found",
                                 f"No video files found in:\n{self.video_dir}\n\n"
                                 f"Supported: {', '.join(sorted(self.extensions))}")
            self.destroy()
            return

        # Load existing annotations (new nested schema)
        self.annotations = self._load_annotations()
        self.metadata = self._load_metadata()
        print("metadata loaded:", len(self.metadata))
        print("sample keys:", list(self.metadata.keys())[:5])

        # State for current video
        self.idx = 0
        self.current_image_tk = None
        self.canvas_image_id = None

        # Original frame size
        self.frame_w = None
        self.frame_h = None

        # Fitted (base) size before applying zoom
        self.base_disp_w = None
        self.base_disp_h = None

        # Final displayed size = base * zoom
        self.disp_w = None
        self.disp_h = None

        # Points (normalized 0..1 relative to current display)
        self.current_points = []
        self.dirty = False
        self.current_events = []
        self.selected_event_id = None

        # NEW: frame id handling
        self.current_frame_id = 1          # last successfully displayed frame (int, 1-based)
        self.frame_id_var = tk.StringVar(value=str(self.current_frame_id))

        # Build UI
        self._build_ui()

        # Load the first video + frame 1 by default
        self._load_current_video(default_frame=True)

        # Keyboard shortcuts
        self.bind("<Key-n>", lambda e: self.on_next())
        self.bind("<Key-p>", lambda e: self.on_prev())
        self.bind("<Key-s>", lambda e: self.on_save())
        self.bind("<Key-r>", lambda e: self.on_remove())
        # Optional: quick delete saved for this frame
        self.bind("<Key-d>", lambda e: self.on_delete_saved())

        # Frame stepping

        self.bind(
            "<Left>",
            lambda e: self._step_frame(-1)
        )
        
        self.bind(
            "<Right>",
            lambda e: self._step_frame(1)
        )
        
        self.bind(
            "<Shift-Left>",
            lambda e: self._step_frame(-10)
        )
        
        self.bind(
            "<Shift-Right>",
            lambda e: self._step_frame(10)
        )

        self.bind(
            "<Control-Left>",
            lambda e: self._step_frame(-100)
        )
        
        self.bind(
            "<Control-Right>",
            lambda e: self._step_frame(100)
        )




    ##add inside class
    def _on_marker_double_click(self, event):

        selected = self.marker_tree.selection()
    
        if not selected:
            return
    
        values = self.marker_tree.item(
            selected[0],
            "values"
        )
    
        frame_id = int(values[2])
    
        self.frame_id_var.set(
            str(frame_id)
        )
    
        self.on_go_frame()
    
    def _parse_timesec(self, text, fps=7):

        if not text:
            return []

        pattern = re.compile(
            r'([A-Za-z]*)?\((\d+:\d+)\)'
        )

        rows = []

        for idx, m in enumerate(
            pattern.finditer(text),
            start=1
        ):

            label = m.group(1) or ""

            mmss = m.group(2)

            mm, ss = mmss.split(":")

            seconds = int(mm) * 60 + int(ss)

            rows.append({

                "idx": idx,

                "label": label,

                "time": mmss,

                "frame": round(seconds * fps)

            })

        return rows
    def _load_metadata(self):

        path = Path(
            "/Users/kesiyun/Desktop/00workspace/00sequenceresult.json"
        )

        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                return json.load(f)

        except Exception as e:

            print(
                "Metadata load error:",
                e
            )

            return {}
    ########
    # ----------------------
    # Files & Data
    # ----------------------
    def _find_videos(self):
        videos = []
        if not self.video_dir.exists() or not self.video_dir.is_dir():
            return videos
        for p in sorted(self.video_dir.iterdir(), key=lambda x: x.name.lower()):
            if p.is_file() and p.suffix.lower() in self.extensions:
                videos.append(p)
        return videos

    def _load_annotations(self):
        if self.json_path.exists():
            try:
                with open(self.json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                cleaned = {}
                # Expect schema: { filename: { frame_str: [[x,y], ...], ... }, ... }
                for fname, frames in data.items():
                    if not isinstance(frames, dict):
                        continue
                    cleaned[fname] = {}
                    for fkey, pts in frames.items():
                        lst = []
                        if isinstance(pts, list):
                            for item in pts:
                                try:
                                    x, y = float(item[0]), float(item[1])
                                    if 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0:
                                        lst.append([x, y])
                                except Exception:
                                    pass
                        cleaned[fname][str(fkey)] = lst
                return cleaned
            except Exception as e:
                messagebox.showwarning("JSON load error",
                                       f"Failed to read existing JSON:\n{self.json_path}\n\n{e}\n\nStarting fresh.")
        return {}

    def _save_annotations(self):
        try:
            with open(self.json_path, "w", encoding="utf-8") as f:
                json.dump(self.annotations, f, indent=2, ensure_ascii=False)
        except Exception as e:
            messagebox.showerror("Save failed", f"Could not save JSON:\n{self.json_path}\n\n{e}")

    # UI

    def _build_ui(self):
        # Filename header
        header_frame = tk.Frame(self)
        header_frame.pack(fill="x", padx=8, pady=(8, 2))
        self.header = tk.Label(header_frame, text="", font=("Arial", 14, "bold"))
        self.header.pack(side="left", anchor="w")

        # Right-side progress
        self.progress_var = tk.StringVar(value="")
        self.progress_label = tk.Label(header_frame, textvariable=self.progress_var, fg="#333")
        self.progress_label.pack(side="right", anchor="e")

        # NEW: Frame ID controls (just below filename)
        frame_ctrl = tk.Frame(self)
        frame_ctrl.pack(fill="x", padx=8, pady=(0, 6))
        tk.Label(frame_ctrl, text="Frame ID (1-based):").pack(side="left")
        self.frame_entry = tk.Entry(frame_ctrl, textvariable=self.frame_id_var, width=10)
        self.frame_entry.pack(side="left", padx=(6, 6))
        self.frame_entry.bind("<Return>", lambda e: self.on_go_frame())
        tk.Button(
            frame_ctrl,
            text="-100",
            command=lambda: self._step_frame(-100)
        ).pack(side="left")
        
        tk.Button(
            frame_ctrl,
            text="-10",
            command=lambda: self._step_frame(-10)
        ).pack(side="left")
        
        tk.Button(
            frame_ctrl,
            text="-1",
            command=lambda: self._step_frame(-1)
        ).pack(side="left")
        self.btn_go = tk.Button(frame_ctrl, text="Go", command=self.on_go_frame, width=6)
        self.btn_go.pack(side="left", padx=(0, 10))
        tk.Button(
            frame_ctrl,
            text="+1",
            command=lambda: self._step_frame(1)
        ).pack(side="left")
        
        tk.Button(
            frame_ctrl,
            text="+10",
            command=lambda: self._step_frame(10)
        ).pack(side="left")
        
        tk.Button(
            frame_ctrl,
            text="+100",
            command=lambda: self._step_frame(100)
        ).pack(side="left")
        self.btn_prev = tk.Button(
            frame_ctrl,
            text="Prev Video",
            command=self.on_prev,
            width=10
        )
        
        self.btn_prev.pack(
            side="left",
            padx=(20, 4)
        )
        
        self.btn_next = tk.Button(
            frame_ctrl,
            text="Next Video",
            command=self.on_next,
            width=10
        )
        
        self.btn_next.pack(
            side="left"
        )

        # Status area
        status_frame = tk.Frame(self)
        status_frame.pack(fill="x", padx=8, pady=(0, 6))
        self.status_var = tk.StringVar(value="")
        self.status_label = tk.Label(status_frame, textvariable=self.status_var, fg="#666")
        self.status_label.pack(side="left")

        # Zoom control
        zoom_frame = tk.Frame(self)
        zoom_frame.pack(fill="x", padx=8, pady=(0, 6))
        tk.Label(zoom_frame, text="Zoom (% of fit):").pack(side="left")
        self.zoom_scale = tk.Scale(
            zoom_frame, from_=20, to=150, orient="horizontal",
            showvalue=True, length=260, command=self._on_zoom_change
        )
        self.zoom_scale.set(int(self.zoom_var.get() * 100))
        self.zoom_scale.pack(side="left", padx=(8, 12))
        tk.Label(zoom_frame, text="Tip: reduce to force a smaller canvas").pack(side="left", padx=(0, 8))

        # Main split area

        main_frame = tk.Frame(self)
        main_frame.pack(fill="both", expand=True)
        
        # -------------------
        # LEFT
        # -------------------
        
        left_frame = tk.Frame(main_frame)
        left_frame.pack(side="left", fill="both", expand=True)
        
        self.canvas = tk.Canvas(
            left_frame,
            bg="#000000",
            highlightthickness=0,
            cursor="crosshair"
        )
        
        self.canvas.pack(
            fill="both",
            expand=True,
            padx=8,
            pady=8
        )
        
        self.canvas.bind(
            "<Button-1>",
            self.on_canvas_click
        )
        
        # -------------------
        # RIGHT
        # -------------------
        
        right_frame = tk.Frame(
            main_frame,
            width=350
        )
        
        right_frame.pack(
            side="right",
            fill="y",
            padx=5
        )
        
        self.right_frame = right_frame
        
        # placeholder labels for now
        
        tk.Label(
            right_frame,
            text="Initial Markers",
            font=("Arial", 12, "bold")
        ).pack(anchor="w")
        
        self.marker_tree = ttk.Treeview(
            right_frame,
            columns=("idx", "label", "frame"),
            show="headings",
            height=5
        )
        
        self.marker_tree.heading("idx", text="#")
        self.marker_tree.heading("label", text="Marker")
        self.marker_tree.heading("frame", text="Frame")
        
        self.marker_tree.column("idx", width=50)
        self.marker_tree.column("label", width=80)
        self.marker_tree.column("frame", width=100)
        
        self.marker_tree.pack(
            fill="x",
            pady=(0,20)
        )
        self.marker_tree.bind(
            "<Double-1>",
            self._on_marker_double_click
        )
        
        
        tk.Label(
            right_frame,
            text="Events",
            font=("Arial", 12, "bold")
        ).pack(anchor="w")
        
        ###
        self.event_tree = ttk.Treeview(
            right_frame,
            columns=("id", "label", "start", "end"),
            show="headings",
            height=5
        )
        
        self.event_tree.heading("id", text="ID")
        self.event_tree.heading("label", text="Label")
        self.event_tree.heading("start", text="Start")
        self.event_tree.heading("end", text="End")
        
        self.event_tree.column("id", width=80)
        self.event_tree.column("label", width=60)
        self.event_tree.column("start", width=70)
        self.event_tree.column("end", width=70)
        
        self.event_tree.pack(
            fill="x",
            pady=(0, 15)
        )
        ###
        self.event_tree.bind(
            "<Double-1>",
            self._on_event_double_click
        )
        ###
        tk.Label(
            right_frame,
            text="Event Editor",
            font=("Arial", 12, "bold")
        ).pack(anchor="w")
        
        tk.Label(
            right_frame,
            text="Label"
        ).pack(anchor="w")
        
        self.event_label_var = tk.StringVar()
        
        tk.Entry(
            right_frame,
            textvariable=self.event_label_var
        ).pack(fill="x")
        
        tk.Label(
            right_frame,
            text="Start Frame"
        ).pack(anchor="w")
        
        self.event_start_var = tk.StringVar()
        
        tk.Entry(
            right_frame,
            textvariable=self.event_start_var
        ).pack(fill="x")
        
        tk.Label(
            right_frame,
            text="End Frame"
        ).pack(anchor="w")
        
        self.event_end_var = tk.StringVar()
        
        tk.Entry(
            right_frame,
            textvariable=self.event_end_var
        ).pack(fill="x")
        ###
        ###
        tk.Button(
            right_frame,
            text="Use Current As Start",
            command=lambda:
                self.event_start_var.set(
                    str(self.current_frame_id)
                )
        ).pack(fill="x", pady=(5,0))
        
        tk.Button(
            right_frame,
            text="Use Current As End",
            command=lambda:
                self.event_end_var.set(
                    str(self.current_frame_id)
                )
        ).pack(fill="x")
        ###
        ###
        tk.Button(
            right_frame,
            text="Add Event",
            command=self._add_event
        ).pack(fill="x", pady=(10,0))
        ###
        tk.Button(
            right_frame,
            text="Update Selected Event",
            command=self._update_event
        ).pack(fill="x", pady=(5,0))
        tk.Button(
            right_frame,
            text="Delete Selected Event",
            command=self._delete_event
        ).pack(fill="x", pady=(5,0))

        ###



    # Navigation & Loading

    def _video_key(self, path: Path):
        return path.stem

    def _compute_fit_size(self, w, h):
        """
        Compute base display size that fits within available screen space (no upscaling),
        then will be optionally adjusted by zoom.
        """
        # Fit to available screen region
        fit_scale = min(self.avail_w / float(w), self.avail_h / float(h), self.fit_scale_cap)
        fit_scale = min(fit_scale, 1.0)  # never upscale on fit
        base_w = max(self.min_disp_w, int(w * fit_scale))
        base_h = max(self.min_disp_h, int(h * fit_scale))

        # Recheck to ensure we didn't exceed available after min clamp
        corr_scale = min(self.avail_w / float(base_w), self.avail_h / float(base_h), 1.0)
        base_w = int(base_w * corr_scale)
        base_h = int(base_h * corr_scale)
        return max(1, base_w), max(1, base_h)

    def _apply_zoom(self, base_w, base_h):
        z = float(self.zoom_var.get())
        disp_w = max(1, int(base_w * z))
        disp_h = max(1, int(base_h * z))
        return disp_w, disp_h

    def _load_current_video(self, default_frame=True):
        """Load current video; if default_frame=True, reset frame to 1; else keep existing frame id."""
        if not (0 <= self.idx < len(self.video_list)):
            return
        path = self.video_list[self.idx]
        key = self._video_key(path)

        self.header.config(text=f"{path.name}")
        self.progress_var.set(f"{self.idx+1} / {len(self.video_list)}")
        
        self._refresh_marker_tree()

        self._load_video_events()
        
        self._refresh_event_tree()
        self._clear_event_editor()

        # Decide which frame to show
        if default_frame:
            target = 1
        else:
            # Keep current chosen id, fallback to 1 if not set
            try:
                target = int(self.frame_id_var.get().strip())
                if target < 1:
                    target = 1
            except Exception:
                target = 1

        # Try to load that frame
        ok = self._load_frame_internal(path, target)
        if not ok:
            # Gracefully fallback to previously good (if any), else try frame 1
            if self.current_image_tk is None:
                # Try frame 1 once
                if target != 1:
                    if not self._load_frame_internal(path, 1):
                        messagebox.showerror("Frame error", f"Failed to load video: {path.name}")
                        self._advance(1)
                        return
            # keep showing whatever is displayed
        # ensure UI reflects the (last valid) frame id
        self.frame_id_var.set(str(self.current_frame_id))
        self._update_status()

    def _advance(self, step):
        new_idx = self.idx + step
        new_idx = min(max(new_idx, 0), len(self.video_list) - 1)
        self.idx = new_idx
        # Per requirement, default to first frame on video change
        self.current_frame_id = 1
        self.frame_id_var.set(str(self.current_frame_id))
        self._load_current_video(default_frame=True)

    def on_next(self):
        self._advance(1)

    def on_prev(self):
        self._advance(-1)

    def _step_frame(self, delta):

        try:

            current = int(
                self.frame_id_var.get()
            )

        except Exception:

            current = self.current_frame_id

        target = max(
            1,
            current + delta
        )

        self.frame_id_var.set(
            str(target)
        )

        self.on_go_frame()
        
    def on_go_frame(self):
        """Triggered by Go button or Enter key"""
        # Parse intended target
        try:
            target = int(self.frame_id_var.get().strip())
        except Exception:
            # Invalid input: keep previous frame
            self.frame_id_var.set(str(self.current_frame_id))
            self._set_status(f"Invalid frame ID input.")
            return

        if target < 1:
            self.frame_id_var.set(str(self.current_frame_id))
            self._set_status(f"Frame {target} is invalid (must be >= 1).")
            return

        path = self.video_list[self.idx]
        ok = self._load_frame_internal(path, target)
        if not ok:
            # keep previous image; revert entry to last valid id
            self.frame_id_var.set(str(self.current_frame_id))
            self._set_status(f"Frame {target} is invalid or unreadable.")
        else:
            self.frame_id_var.set(str(self.current_frame_id))
            self._set_status(f"Showing frame {self.current_frame_id}")

    # Core frame loading + canvas draw

    def _load_frame_internal(self, path: Path, frame_id_1based: int) -> bool:
        """Returns True if frame loaded and displayed; False if failed (keeps previous display)."""
        try:
            frame = read_frame(path, frame_id_1based)  # numpy array HxWx3 RGB
        except Exception:
            return False

        self.frame_h, self.frame_w = int(frame.shape[0]), int(frame.shape[1])

        # Base fit size (fits screen)
        self.base_disp_w, self.base_disp_h = self._compute_fit_size(self.frame_w, self.frame_h)

        # Final displayed size after zoom
        self.disp_w, self.disp_h = self._apply_zoom(self.base_disp_w, self.base_disp_h)

        # Prepare image for Tk
        img = Image.fromarray(frame)
        if (self.disp_w, self.disp_h) != (self.frame_w, self.frame_h):
            img = img.resize((self.disp_w, self.disp_h), Image.LANCZOS)
        self.current_image_tk = ImageTk.PhotoImage(img)

        # Configure canvas
        self.canvas.config(width=self.disp_w, height=self.disp_h)
        self.canvas.delete("all")
        self.canvas_image_id = self.canvas.create_image(0, 0, image=self.current_image_tk, anchor="nw")

        # Load existing points for this video + frame (normalized)
        self._load_points_for_current(path, frame_id_1based)

        # Set current frame id to the successfully displayed one
        self.current_frame_id = int(frame_id_1based)

        self.dirty = False
        self._redraw_points()
        return True

    def _load_points_for_current(self, path: Path, frame_id_1based: int):
        key = self._video_key(path)
        fkey = str(int(frame_id_1based))  # string keys by decision
        pts = []
        if key in self.annotations and isinstance(self.annotations[key], dict):
            pts = self.annotations[key].get(fkey, [])
        self.current_points = []
        if isinstance(pts, list):
            for p in pts:
                try:
                    x, y = float(p[0]), float(p[1])
                    if 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0:
                        self.current_points.append((x, y))
                except Exception:
                    pass

    # Interaction

    def _on_zoom_change(self, val):
        # Update zoom variable from slider (percent to 0.x)
        try:
            z = max(0.2, min(1.5, float(val) / 100.0))
        except Exception:
            z = 1.0
        self.zoom_var.set(z)
        self._redraw_with_new_zoom()

    def _redraw_with_new_zoom(self):
        # Recompute displayed size from base size and zoom
        if self.base_disp_w is None or self.base_disp_h is None:
            return
        self.disp_w, self.disp_h = self._apply_zoom(self.base_disp_w, self.base_disp_h)

        # Re-read the current frame to regenerate display at new size
        path = self.video_list[self.idx]
        try:
            frame = read_frame(path, self.current_frame_id)
        except Exception as e:
            messagebox.showerror("Frame error", f"Failed to re-read frame\n{path}\n\n{e}")
            return

        img = Image.fromarray(frame)
        if (self.disp_w, self.disp_h) != (self.frame_w, self.frame_h):
            img = img.resize((self.disp_w, self.disp_h), Image.LANCZOS)
        self.current_image_tk = ImageTk.PhotoImage(img)

        # Update canvas
        self.canvas.config(width=self.disp_w, height=self.disp_h)
        self.canvas.delete("all")
        self.canvas_image_id = self.canvas.create_image(0, 0, image=self.current_image_tk, anchor="nw")
        self._redraw_points()

    def on_canvas_click(self, event):
        if self.disp_w is None or self.disp_h is None:
            return
        x_disp, y_disp = event.x, event.y
        x_disp = max(0, min(self.disp_w, x_disp))
        y_disp = max(0, min(self.disp_h, y_disp))
        x_norm = x_disp / float(self.disp_w)
        y_norm = y_disp / float(self.disp_h)
        self.current_points.append((x_norm, y_norm))
        self.dirty = True
        self._update_status()
        self._redraw_points()

    def on_remove(self):
        """Clear current working points (does not touch saved JSON until Save)."""
        if self.current_points:
            self.current_points = []
            self.dirty = True
            self._update_status()
            self._redraw_points()

    def on_delete_saved(self):
        """Delete saved points for this video+frame from JSON file."""
        path = self.video_list[self.idx]
        key = self._video_key(path)
        fkey = str(self.current_frame_id)
        if key in self.annotations and fkey in self.annotations.get(key, {}):
            # Optional confirm
            if messagebox.askyesno("Delete saved", f"Delete saved points for {key} frame {fkey}?"):
                try:
                    del self.annotations[key][fkey]
                    # If frame dict becomes empty, remove the video key as well
                    if not self.annotations[key]:
                        del self.annotations[key]
                    self._save_annotations()
                    # Also clear working set in UI to reflect deletion
                    self.current_points = []
                    self.dirty = False
                    self._update_status()
                    self._redraw_points()
                    self._set_status(f"Deleted saved points for {key} frame {fkey}.")
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to delete saved points:\n{e}")
        else:
            self._set_status("No saved points for this frame to delete.")

    def on_save(self):
        path = self.video_list[self.idx]
        key = self._video_key(path)
        fkey = str(self.current_frame_id)
        if key not in self.annotations or not isinstance(self.annotations[key], dict):
            self.annotations[key] = {}
        self.annotations[key][fkey] = [[float(x), float(y)] for (x, y) in self.current_points]
        self._save_annotations()
        self.dirty = False
        self._update_status()

    # Drawing Helpers

    def _redraw_points(self):
        self.canvas.delete("marker")
        if self.disp_w is None or self.disp_h is None:
            return
        r = max(3, int(max(self.disp_w, self.disp_h) * 0.006))
        for i, (x_norm, y_norm) in enumerate(self.current_points, start=1):
            x = x_norm * self.disp_w
            y = y_norm * self.disp_h
            self.canvas.create_oval(x - r, y - r, x + r, y + r,
                                    fill="#ff5252", outline="white", width=1.5, tags="marker")
            self.canvas.create_text(x + 10, y - 10, text=str(i),
                                    fill="yellow", font=("Arial", 11, "bold"), tags="marker")
    ###inside clas?
    def _on_event_double_click(self, event):

        selected = self.event_tree.selection()

        if not selected:
            return

        values = self.event_tree.item(
            selected[0],
            "values"
        )

        event_id = values[0]

        self.selected_event_id = event_id

        self.event_label_var.set(
            values[1]
        )

        self.event_start_var.set(
            values[2]
        )

        self.event_end_var.set(
            values[3]
        )

        start_frame = int(values[2])

        self.frame_id_var.set(
            str(start_frame)
        )

        self.on_go_frame()
    ###
    def _delete_event(self):

        selected = self.event_tree.selection()

        if not selected:

            self._set_status(
                "No event selected"
            )

            return

        item_id = selected[0]

        values = self.event_tree.item(
            item_id,
            "values"
        )

        event_id = values[0]

        self.event_tree.delete(
            item_id
        )

        self.current_events = [

            e

            for e in self.current_events

            if e["id"] != event_id

        ]
        self._save_video_events()
        self._clear_event_editor()

        self._set_status(
            f"Deleted {event_id}"
        )
    ###
    def _next_event_id(self):

        if not self.current_events:
            return "evt_001"

        max_id = 0

        for event in self.current_events:

            try:

                num = int(
                    event["id"].split("_")[1]
                )

                max_id = max(
                    max_id,
                    num
                )

            except Exception:

                pass

        return f"evt_{max_id + 1:03d}"
    def _update_event(self):

        if self.selected_event_id is None:

            self._set_status(
                "No event selected"
            )

            return

        try:

            start_frame = int(
                self.event_start_var.get()
            )

            end_frame = int(
                self.event_end_var.get()
            )

        except Exception:

            self._set_status(
                "Invalid start/end frame"
            )

            return

        for event in self.current_events:

            if event["id"] == self.selected_event_id:

                event["label"] = (
                    self.event_label_var.get().strip()
                )

                event["start_frame"] = (
                    start_frame
                )

                event["end_frame"] = (
                    end_frame
                )

                break

        self._save_video_events()

        self._refresh_event_tree()

        self._set_status(
            f"Updated {self.selected_event_id}"
        )
    def _add_event(self):

        try:

            start_frame = int(
                self.event_start_var.get()
            )

            end_frame = int(
                self.event_end_var.get()
            )

        except Exception:

            self._set_status(
                "Invalid start/end frame"
            )

            return

        event_id = self._next_event_id()

        event = {

            "id": event_id,

            "label": self.event_label_var.get().strip(),

            "start_frame": start_frame,

            "end_frame": end_frame

        }

        self.current_events.append(event)
        self._save_video_events()
        self._clear_event_editor()

        self.event_tree.insert(

            "",

            "end",

            values=(

                event["id"],

                event["label"],

                event["start_frame"],

                event["end_frame"]

            )
        )

        self._set_status(
            f"Added {event_id}"
        )

    ###
    def _clear_event_editor(self):

        self.selected_event_id = None

        self.event_label_var.set("")

        self.event_start_var.set("")

        self.event_end_var.set("")
    ###

    def _refresh_event_tree(self):

        self.event_tree.delete(
            *self.event_tree.get_children()
        )

        for event in self.current_events:

            self.event_tree.insert(

                "",

                "end",

                values=(

                    event["id"],

                    event["label"],

                    event["start_frame"],

                    event["end_frame"]

                )
            )

    def _event_json_path(self):

        video_path = self.video_list[self.idx]

        return video_path.with_suffix(".json")
        
    def _load_video_events(self):

        path = self._event_json_path()

        self.current_events = []

        if not path.exists():
            return

        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            self.current_events = data.get(
                "events",
                []
            )

        except Exception as e:

            print(
                "Event load error:",
                e
            )
    def _save_video_events(self):

        path = self._event_json_path()

        payload = {

            "video": self._video_key(
                self.video_list[self.idx]
            ),

            "events": self.current_events
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                payload,
                f,
                indent=2,
                ensure_ascii=False
            )
    def _refresh_marker_tree(self):

        self.marker_tree.delete(
            *self.marker_tree.get_children()
        )

        video_key = self.video_list[
            self.idx
        ].stem
        print(
            "video_key:",
            video_key
        )
        print(
            "metadata exists:",
            video_key in self.metadata
        )

        info = self.metadata.get(
            video_key,
            {}
        )

        markers = self._parse_timesec(
            info.get(
                "timesec",
                ""
            )
        )

        for item in markers:

            self.marker_tree.insert(

                "",

                "end",

                values=(

                    f"{item['idx']:03d}",

                    item["label"]
                    if item["label"]
                    else "[none]",

                    item["frame"]

                )
            )

        
        ###
    def _update_status(self):
        path = self.video_list[self.idx]
        key = self._video_key(path)
        fkey = str(self.current_frame_id)
        count = len(self.current_points)
        saved = False
        saved_pts = []
        if key in self.annotations:
            saved_pts = self.annotations.get(key, {}).get(fkey, [])
        if isinstance(saved_pts, list) and len(saved_pts) == count:
            try:
                saved = all(
                    abs(saved_pts[i][0] - self.current_points[i][0]) < 1e-9 and
                    abs(saved_pts[i][1] - self.current_points[i][1]) < 1e-9
                    for i in range(count)
                )
            except Exception:
                saved = False
        if self.dirty and not saved:
            self.status_var.set(f"{self._status_prefix()}  •  *Not saved*")
        else:
            # Clarify whether this frame currently has any saved points
            has_saved = isinstance(saved_pts, list) and len(saved_pts) > 0
            suffix = "Saved" if has_saved and saved else ("No saved points" if not has_saved else "Saved (differs)")
            self.status_var.set(f"{self._status_prefix()}  •  {suffix}")

    def _status_prefix(self):
        return f"Video {self.idx+1}/{len(self.video_list)}  •  Frame {self.current_frame_id}  •  Points: {len(self.current_points)}"

    def _set_status(self, text):
        self.status_var.set(text)

# 1) Set your video directory here:

#VIDEO_DIR = r'/home/ubuntu-user/Desktop/portable/sample_mp4videos'
#VIDEO_DIR = r'/Volumes/KINGSTON/portable/sample_mp4videos'
#VIDEO_DIR = r'/Volumes/Green SSD/00Workspace_portable/videos'
VIDEO_DIR = r'/Users/kesiyun/Desktop/00workspace/videos'


# If a previous instance exists in the notebook, close it cleanly
try:
    _ = app
    import tkinter as _tk
    if isinstance(app, _tk.Tk):
        app.destroy()
except Exception:
    pass


app = VideoAnnotator(
    VIDEO_DIR,
    margin=(80, 220),          # tweak if menu bar/dock uses more space
    min_display_size=(320, 240),
    fit_scale_cap=1.0          # never upscale on fit; only zoom slider can upscale
)
try:
    app.mainloop()
except Exception as e:
    print("Error running Tkinter main loop:", e)
    traceback.print_exc(file=sys.stdout)



metadata loaded: 57
sample keys: ['1A', '1B', '1C', '1D', '1E']
video_key: 11A
metadata exists: True
